# Next-Hour Bike-Sharing Demand Forecasting and High-Demand Alerts
**ECE 539 · Fall 2026 · Team G02**

## Project Overview
This project aims to help bike-sharing operations planners anticipate
next-hour rental demand and identify high-demand periods. We use the
UCI Bike Sharing Dataset, which contains hourly rental counts,
environmental observations, and calendar information.

## System Pipeline
Our system consists of three connected components:

1. **Environmental Forecasting (NN 1):**
   Uses the previous three hours of temperature, humidity, and wind speed
   to predict these conditions for the next hour.

2. **Rental-Demand Forecasting (NN 2):**
   Combines the predicted environmental conditions, the previous hour’s
   rental count, and target-hour calendar features to predict next-hour
   total rentals.

3. **Demand Analysis and Alerts (Non-neural):**
   Classifies predicted demand as low, medium, or high using thresholds
   derived from training data and flags high-demand hours. It also
   summarizes historical average demand by hour.

**Pipeline:** Historical environment → NN 1 → Predicted environment
→ NN 2 + recent rental count and calendar features → Predicted rentals
→ Demand level and alert

## Main Research Question
Does forecasting next-hour environmental conditions improve rental-demand
prediction compared with using the latest observed conditions?

We compare standard and residual MLPs with an environmental persistence
baseline. We also evaluate how replacing predicted conditions with the
latest observations affects a fixed, trained demand model.

## 1. Environment Setup

In [1]:
%pip install torch numpy pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    f1_score,
)

# Set random seeds to improve reproducibility.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select the available device.
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
print(f"Random seed: {SEED}")

PyTorch version: 2.15.0
Device: mps
Random seed: 42



## 2. Data Preparation

We load the hourly bike-sharing data and construct timestamps.
For each target hour, we collect temperature, humidity, and wind speed
from the preceding three hours as inputs. The target is the environmental
conditions recorded at that hour.

Windows with missing hours are excluded to ensure consecutive observations.

In [3]:
from pathlib import Path

# 1. Load and sort the original data.
df = pd.read_csv(Path("../hour.csv"))

df["timestamp"] = (
    pd.to_datetime(df["dteday"])
    + pd.to_timedelta(df["hr"], unit="h")
)

df = df.sort_values("timestamp").reset_index(drop=True)

if df["timestamp"].duplicated().any():
    raise ValueError("Duplicate hourly timestamps found.")

# 2. Create a shared table.
# Each row represents the target hour we want to predict.
env_cols = ["temp", "hum", "windspeed"]
calendar_cols = ["hr", "mnth", "weekday", "workingday", "holiday"]

model_data = df[
    ["timestamp"] + env_cols + calendar_cols + ["cnt"]
].copy()

# 3. Add the preceding three hours of environmental observations.
nn1_input_cols = []

for lag in [3, 2, 1]:
    for col in env_cols:
        lag_col = f"{col}_lag{lag}"
        model_data[lag_col] = df[col].shift(lag)
        nn1_input_cols.append(lag_col)

# 4. Add the previous hour's rental count for NN 2.
model_data["cnt_lag1"] = df["cnt"].shift(1)

# 5. Keep only windows with three consecutive preceding hours.
valid_window = pd.Series(True, index=df.index)

for lag in [1, 2, 3]:
    valid_window &= (
        df["timestamp"] - df["timestamp"].shift(lag)
        == pd.Timedelta(hours=lag)
    )

model_data = (
    model_data.loc[valid_window]
    .dropna()
    .set_index("timestamp")
    .sort_index()
)

# 6. Define the inputs and targets.
nn1_target_cols = env_cols

# These prediction columns will be added AFTER running NN 1.
pred_env_cols = ["pred_temp", "pred_hum", "pred_windspeed"]

# Calendar features are still unencoded at this stage.
nn2_input_cols = pred_env_cols + ["cnt_lag1"] + calendar_cols
nn2_target_col = "cnt"

print(f"Original records: {len(df)}")
print(f"Valid forecasting samples: {len(model_data)}")

display(model_data.head())

Original records: 17379
Valid forecasting samples: 17153


,temp,hum,windspeed,hr,mnth,weekday,workingday,holiday,cnt,temp_lag3,hum_lag3,windspeed_lag3,temp_lag2,hum_lag2,windspeed_lag2,temp_lag1,hum_lag1,windspeed_lag1,cnt_lag1
timestamp,,,,,,,,,,,,,,,,,,,
2011-01-01 03:00:00,0.24,0.75,0.0000,3,1,6,0,0,13,0.24,0.81,0.0,0.22,0.80,0.0000,0.22,0.80,0.0000,32.0
2011-01-01 04:00:00,0.24,0.75,0.0000,4,1,6,0,0,1,0.22,0.80,0.0,0.22,0.80,0.0000,0.24,0.75,0.0000,13.0
2011-01-01 05:00:00,0.24,0.75,0.0896,5,1,6,0,0,1,0.22,0.80,0.0,0.24,0.75,0.0000,0.24,0.75,0.0000,1.0
2011-01-01 06:00:00,0.22,0.80,0.0000,6,1,6,0,0,2,0.24,0.75,0.0,0.24,0.75,0.0000,0.24,0.75,0.0896,1.0
2011-01-01 07:00:00,0.20,0.86,0.0000,7,1,6,0,0,3,0.24,0.75,0.0,0.24,0.75,0.0896,0.22,0.80,0.0000,2.0


## Part 1: Environmental Forecasting

This component predicts next-hour temperature, humidity, and wind speed
using observations from the preceding three consecutive hours.

### Inputs and Targets
- **Input:** Three hourly observations of three environmental variables,
  flattened into a vector of 9 values.
- **Target:** The next hour’s recorded temperature, humidity, and wind speed
  (3 values).

For example, observations at 00:00, 01:00, and 02:00 are used to predict
conditions at 03:00.

### Methods
We will compare three approaches:

1. **Persistence baseline:** Use the latest observed conditions as the
   next-hour prediction.
2. **Standard MLP:** Directly predict the three next-hour environmental values.
3. **Residual MLP:** Predict changes and add them to the latest observations:

   predicted next-hour conditions = latest conditions + predicted changes